In [ ]:
import math
import random
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from pathlib import Path

SEED = 7
DATA_DIR = Path('shakespeare-dataset-main/text')
BLOCK_SIZE, BATCH_SIZE, NUM_WORKERS = 64, 128, 0
DIM, NUM_LAYERS, NUM_HEADS, FF_DIM, DROPOUT = 128, 6, 4, 512, 0.1
DIFFUSION_STEPS, BETA_START, BETA_END = 1000, 1e-4, 0.02
LEARNING_RATE, WEIGHT_DECAY, GRAD_CLIP_NORM = 3e-4, 1e-2, 1.0
TRAIN_STEPS, LOG_EVERY, SAMPLE_EVERY = 10_000, 50, 1_000
ROUNDING_WEIGHT, COVERAGE_WEIGHT, CONTAINMENT_WEIGHT, PRIOR_SAMPLES = 1.0, 1.0, 1.0, 1_024
SAMPLE_BATCH_SIZE, SAMPLE_REVERSE_STEPS = 2, 250

device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
random.seed(SEED)
print(f'Using {device}')


In [ ]:
def clean_folger_text(raw):
    lines = raw.replace('\r\n', '\n').replace('\r', '\n').splitlines(keepends=True)
    return ''.join(lines[7:]).lstrip('\n')

paths = sorted(DATA_DIR.glob('*.txt'))
if not paths:
    raise FileNotFoundError(f'No .txt files in {DATA_DIR}')
text = '\n\n'.join(clean_folger_text(p.read_text(encoding='utf-8')) for p in paths)
chars = sorted(set(text))
stoi = {char: index for index, char in enumerate(chars)}
itos = dict(enumerate(chars))
tokens = torch.tensor([stoi[c] for c in text], dtype=torch.long)

class CharacterBlocks(Dataset):
    def __init__(self, values, block_size):
        self.values = values
        self.block_size = block_size
    def __len__(self): return len(self.values) // self.block_size
    def __getitem__(self, index):
        start = index * self.block_size
        return self.values[start:start + self.block_size]

dataset = CharacterBlocks(tokens, BLOCK_SIZE)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=NUM_WORKERS)
print(f'Loaded {len(paths)} works | {len(text):,} characters | vocab={len(chars)} | batch={tuple(next(iter(loader)).shape)}')


In [ ]:
# Gaussian embedding
class GaussianEmbedding(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.mu = nn.Parameter(torch.randn(vocab_size, dim) * .02)
        initial_std = F.softplus(torch.randn(vocab_size, dim) * 0.25)

        # Initiatilize with standard aggregate variance
        target_std_rms = (1 - self.mu.detach().square().mean()).sqrt()
        initial_std = initial_std * target_std_rms / initial_std.square().mean().sqrt()
        self.raw_std = nn.Parameter(torch.log(torch.expm1(initial_std)))
        self.mixture_logits = nn.Parameter(torch.zeros(vocab_size))
    @property
    def std(self): return F.softplus(self.raw_std)
    @property
    def mixture_weights(self): return self.mixture_logits.softmax(0)
    def sample(self, ids):
        mean = self.mu[ids]
        return mean + self.std[ids] * torch.randn_like(mean)
    def squared_distances(self, vectors):
        return vectors.square().sum(-1, keepdim=True) + self.mu.square().sum(-1) - 2 * vectors @ self.mu.T

# Model 
class SimpleGaussianDiffusionLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = GaussianEmbedding(vocab_size, DIM)
        self.position = nn.Embedding(BLOCK_SIZE, DIM)
        frequencies = torch.exp(-math.log(10_000) * torch.arange(DIM // 2) / (DIM // 2))
        self.register_buffer('time_frequencies', frequencies)
        layer = nn.TransformerEncoderLayer(DIM, NUM_HEADS, FF_DIM, DROPOUT, batch_first=True, norm_first=False, activation='gelu')
        self.transformer = nn.TransformerEncoder(layer, NUM_LAYERS)
        self.final_norm, self.x0_head = nn.LayerNorm(DIM), nn.Linear(DIM, DIM)
    def forward(self, x_t, timesteps):
        pos = torch.arange(x_t.size(1), device=x_t.device)
        angles = timesteps[:, None] * self.time_frequencies
        time_embedding = torch.cat((angles.sin(), angles.cos()), dim=-1)
        hidden = x_t + self.position(pos).unsqueeze(0) + time_embedding.unsqueeze(1)
        return self.x0_head(self.final_norm(self.transformer(hidden)))
    def rounding_logits(self, vectors): return -self.embedding.squared_distances(vectors)
    def round_to_tokens(self, vectors): return self.rounding_logits(vectors).argmax(-1)

model = SimpleGaussianDiffusionLM(len(chars)).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')


## Diffusion objective

Loss is denoising MSE plus character-rounding cross-entropy plus a sampled bidirectional KL between the token-Gaussian mixture and the standard-normal prior. Mixture weights and component variances are learned, so tokens can occupy different amounts of prior mass.


In [ ]:
betas = torch.linspace(BETA_START, BETA_END, DIFFUSION_STEPS, device=device)
alpha_bar = torch.cat((torch.ones(1, device=device), torch.cumprod(1 - betas, 0)))

def q_sample(x0, t):
    alpha = alpha_bar[t].view(-1, 1, 1)
    return alpha.sqrt() * x0 + (1 - alpha).sqrt() * torch.randn_like(x0)

@torch.no_grad()
def aggregate_statistics():
    probabilities = model.embedding.mixture_weights
    means = model.embedding.mu
    variances = model.embedding.std.square()
    aggregate_mean = (probabilities[:, None] * means).sum(dim=0)
    aggregate_second_moment = (probabilities[:, None] * (means.square() + variances)).sum(dim=0)
    aggregate_variance = aggregate_second_moment - aggregate_mean.square()
    return aggregate_mean.mean(), aggregate_variance.mean()

def standard_log_prob(values):
    return -.5 * (values.square() + math.log(2 * math.pi)).sum(-1)

def mixture_log_prob(values):
    std = model.embedding.std
    scaled = (values[:, None] - model.embedding.mu) / std
    component_log_prob = -.5 * (scaled.square() + 2 * std.log() + math.log(2 * math.pi)).sum(-1)
    return torch.logsumexp(component_log_prob + model.embedding.mixture_logits.log_softmax(0), dim=-1)

def aggregate_kl_loss():
    count = max(1, PRIOR_SAMPLES // len(chars))
    component_ids = torch.arange(len(chars), device=device).repeat_interleave(count)
    from_mixture = model.embedding.sample(component_ids)
    from_prior = torch.randn(PRIOR_SAMPLES, DIM, device=device)
    coverage = (standard_log_prob(from_prior) - mixture_log_prob(from_prior)).mean()
    log_ratio = mixture_log_prob(from_mixture) - standard_log_prob(from_mixture)
    containment = (model.embedding.mixture_weights * log_ratio.reshape(len(chars), count).mean(1)).sum()
    return COVERAGE_WEIGHT * coverage + CONTAINMENT_WEIGHT * containment
def loss_terms(ids):
    x0 = model.embedding.sample(ids)
    t = torch.randint(1, DIFFUSION_STEPS + 1, (ids.size(0),), device=device)
    prediction = model(q_sample(x0, t), t)
    mse = F.mse_loss(prediction, x0)
    rounding = F.cross_entropy(model.rounding_logits(x0).flatten(0, 1), ids.flatten())
    weighted_prior = aggregate_kl_loss()
    weighted_ce = ROUNDING_WEIGHT * rounding
    total = mse + weighted_ce + weighted_prior
    return total, mse, weighted_ce, weighted_prior
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


In [ ]:
def step(batch):
    optimizer.zero_grad(set_to_none=True)
    values = loss_terms(batch.to(device))
    values[0].backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
    optimizer.step()
    aggregate_mean, aggregate_variance = aggregate_statistics()
    return *values, aggregate_mean, aggregate_variance, grad_norm
def train(steps=TRAIN_STEPS):
    model.train()
    iterator = iter(loader)
    running = torch.zeros(7)
    for number in range(1, steps + 1):
        try: batch = next(iterator)
        except StopIteration:
            iterator = iter(loader)
            batch = next(iterator)
        values = step(batch)
        running += torch.tensor([value.item() for value in values])
        if number % LOG_EVERY == 0 or number == steps:
            mean = running / min(LOG_EVERY, number)
            print(
                f'step {number}/{steps} | total={mean[0]:.4f} | '
                f'mse={mean[1]:.4f} | weighted_ce={mean[2]:.4f} | weighted_prior={mean[3]:.6f} | '
                f'aggregate_mean={mean[4]:.4f} | aggregate_variance={mean[5]:.4f} | '
                f'grad_norm={mean[6]:.4f}'
            )
            running.zero_()
        if number % SAMPLE_EVERY == 0:
            print(f'\nSamples at step {number}:')
            for token_block in sample().cpu():
                print(''.join(itos[token.item()] for token in token_block))
                print()


In [ ]:
@torch.no_grad()
def sample(batch_size=SAMPLE_BATCH_SIZE, reverse_steps=SAMPLE_REVERSE_STEPS):
    was_training = model.training
    model.eval()
    x_t = torch.randn(batch_size, BLOCK_SIZE, DIM, device=device)
    times = torch.linspace(DIFFUSION_STEPS, 0, reverse_steps + 1, device=device).long()
    for current, next_time in zip(times[:-1], times[1:]):
        t = torch.full((batch_size,), current, device=device, dtype=torch.long)
        x0_hat = model(x_t, t)
        a_t, a_next = alpha_bar[current], alpha_bar[next_time]
        eps_hat = (x_t - a_t.sqrt() * x0_hat) / (1 - a_t).sqrt().clamp_min(1e-8)
        x_t = a_next.sqrt() * x0_hat + (1 - a_next).sqrt() * eps_hat
    result = model.round_to_tokens(x_t)
    model.train(was_training)
    return result

train()

In [ ]:
for token_block in sample(reverse_steps=250).cpu():
                print(''.join(itos[token.item()] for token in token_block))
                print()

## Denoising diagnostic

Denoise one real text block at every 100th timestep. MSE and character accuracy use the full batch; the text shows its first block. The baseline predicts `sqrt(alpha_bar[t]) * x_t`.


In [ ]:
@torch.no_grad()
def denoising_diagnostic():
    was_training = model.training
    model.eval()
    ids = next(iter(loader)).to(device)
    x0 = model.embedding.sample(ids)
    noise = torch.randn_like(x0)
    print(f'original: {"".join(itos[i] for i in ids[0].tolist())!r}')
    for timestep in range(100, DIFFUSION_STEPS + 1, 100):
        alpha = alpha_bar[timestep]
        x_t = alpha.sqrt() * x0 + (1 - alpha).sqrt() * noise
        t = torch.full((len(ids),), timestep, device=device, dtype=torch.long)
        prediction = model(x_t, t)
        decoded = model.round_to_tokens(prediction)
        mse = F.mse_loss(prediction, x0).item()
        baseline = F.mse_loss(alpha.sqrt() * x_t, x0).item()
        accuracy = (decoded == ids).float().mean().item()
        text = ''.join(itos[i] for i in decoded[0].tolist())
        print(f't={timestep:4d} | mse={mse:.4f} | baseline={baseline:.4f} | accuracy={accuracy:.1%} | {text!r}')
    model.train(was_training)

denoising_diagnostic()
